In [7]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; ++i) {
        x[i] = rand() % 100;
    }
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main() {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    double dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();
    double dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] + b[i]) {
            errors++;
        }
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex1_timed.cu


In [19]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed

/bin/bash: line 1: nvcc: command not found


In [9]:
%%writefile ex2_check.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; ++i) {
        x[i] = rand() % 100;
    }
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main() {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    double dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    double dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    int errors = 0;
    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] + b[i]) {
            errors++;
        }
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return 0;
}

Writing ex2_check.cu


In [20]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check

/bin/bash: line 1: nvcc: command not found


In [11]:
%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define WIDTH 800
#define HEIGHT 600

#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

__global__ void PictureKernel(unsigned char *d_in, unsigned char *d_out, int width, int height) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (col < width && row < height) {
        int idx = row * width + col;
        int temp = d_in[idx] * 2;
        d_out[idx] = (temp > 255) ? 255 : temp;
    }
}

int main() {
    int count = WIDTH * HEIGHT;
    int size = count * sizeof(unsigned char);
    unsigned char *h_in, *h_out, *h_ver;
    unsigned char *d_in, *d_out;
    double dStart, dElaps;

    h_in = (unsigned char *)malloc(size);
    h_out = (unsigned char *)malloc(size);
    h_ver = (unsigned char *)malloc(size);

    for (int i = 0; i < count; i++) {
        h_in[i] = rand() % 256;
    }

    CHECK(cudaMalloc((void **)&d_in, size));
    CHECK(cudaMalloc((void **)&d_out, size));

    CHECK(cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice));

    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks((WIDTH + threadsPerBlock.x - 1) / threadsPerBlock.x, (HEIGHT + threadsPerBlock.y - 1) / threadsPerBlock.y);
    printf("Grid: %d x %d blocks of %d x %d threads (%d blocks)\n", numBlocks.x, numBlocks.y, threadsPerBlock.x, threadsPerBlock.y, numBlocks.x * numBlocks.y);

    dStart = cpuSecond();
    PictureKernel<<<numBlocks, threadsPerBlock>>>(d_in, d_out, WIDTH, HEIGHT);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost));

    for (int i = 0; i < count; i++) {
        int temp = h_in[i] * 2;
        h_ver[i] = (temp > 255) ? 255 : temp;
    }

    int errors = 0;
    int capped = 0;
    for (int i = 0; i < count; i++) {
        if (h_out[i] != h_ver[i]) {
            errors++;
        }
        if (h_out[i] == 255) {
            capped++;
        }
    }

    printf("First ten original values (row 0):\n");
    for (int i = 0; i < 10; i++) printf("%d ", h_in[i]);
    printf("\nFirst ten updated values (row 0):\n");
    for (int i = 0; i < 10; i++) printf("%d ", h_out[i]);
    printf("\n");

    printf("Verification\n");
    printf("%d mismatches out of %d\n", errors, count);
    printf("Pixels capped at 255\n");
    printf("%d\n", capped);
    printf("Time taken - %.7f\n", dElaps);

    free(h_in); free(h_out); free(h_ver);
    CHECK(cudaFree(d_in)); CHECK(cudaFree(d_out));
    return 0;
}

Writing ex3_picture.cu


In [21]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture

/bin/bash: line 1: nvcc: command not found


In [13]:
%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define NX 500
#define NY 500
#define NZ 500

#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

__global__ void mulElem3D(int *a, int *b, int *c, int nx, int ny, int nz) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    if (x < nx && y < ny && z < nz) {
        long long idx = ((long long)z * ny + y) * nx + x;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, long long size) {
    for (long long i = 0; i < size; ++i) {
        x[i] = rand() % 10;
    }
}

int main() {
    long long count = (long long)NX * NY * NZ;
    long long size = count * sizeof(int);
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    double dStart, dElaps;

    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size); random_ints(a, count);
    b = (int *)malloc(size); random_ints(b, count);
    c = (int *)malloc(size);

    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    dim3 threadsPerBlock(8, 8, 8);
    dim3 numBlocks((NX + threadsPerBlock.x - 1) / threadsPerBlock.x, (NY + threadsPerBlock.y - 1) / threadsPerBlock.y, (NZ + threadsPerBlock.z - 1) / threadsPerBlock.z);
    printf("Grid: %d x %d x %d blocks\n", numBlocks.x, numBlocks.y, numBlocks.z);

    dStart = cpuSecond();
    mulElem3D<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, NX, NY, NZ);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    long long startPrint = count - 1000;
    for (long long i = startPrint; i < count; i++) {
        long long temp = i;
        int x = temp % NX;
        temp /= NX;
        int y = temp % NY;
        int z = temp / NY;
        printf("C[%d][%d][%d] = %d x %d = %d\n", z, y, x, a[i], b[i], c[i]);
    }

    long long mismatches = 0;
    for (long long i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) {
            mismatches++;
        }
    }
    printf("Verification: %lld mismatches out of %lld\n", mismatches, count);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return 0;
}

Writing ex4_volume.cu


In [22]:
!nvcc -arch=sm_75 ex4_volume.cu -o ex4_volume && ./ex4_volume

/bin/bash: line 1: nvcc: command not found


In [6]:
with open('lab_instructions.txt', 'r') as f:
    text = f.read()

pages = text.split('=== PAGE ')
for i in range(1, len(pages)):
    page_content = pages[i]
    print(f"==================== PAGE {i} ====================")
    print(page_content)
    print("\n\n")

==================== PAGE 1 ====================
1 ===
BSc (Hons) in Information Technology
 Year 3
Lab Sheet 10 – CUDA (Google Colab Edition)
SE3082 – Parallel Computing
Semester 2, 2026
Introduction
This lab continues Lab 9. You will run everything on a real NVIDIA GPU in Google Colab, using exactly the same
write → compile → run workflow as before. You will learn how to measure how long a GPU program takes, how
to make CUDA programs report their own errors, and how to cover pictures and volumes with two- and
three-dimensional grids.
By the end of this lab you will be able to:
  time a kernel with a CPU timer, and explain why cudaDeviceSynchronize() must come before the clock is
stopped;
 wrap CUDA calls in a CHECK() macro and read the error messages it prints;
 cover an 800 × 600 picture with a 2D grid of 16 × 16 blocks, and a 500 × 500 × 500 volume with a 3D grid;
 write correct bounds checks for one, two and three dimensions.
Exercise
Topic
1
Timing a GPU program
2
Catching er